<a href="https://colab.research.google.com/github/Ritu362/Tuki-Bot/blob/main/Copy_of_Untitled1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install yfinance -q

importimport time
import requests
import pandas as pd
import yfinance as yf

# আপনার টেলিগ্রাম টোকেন এবং চ্যাট আইডি
TOKEN = "9249300034:AAHI1qc2pUlo2G-6FVuygI0VGUAKMdyGuog"
CHAT_ID = "7490703095"

# একসাথে একাধিক জনপ্রিয় পেয়ার ট্র্যাক করার লিস্ট
SYMBOLS = ["EURUSD=X", "GBPUSD=X", "USDJPY=X"]

def send_telegram_signal(message):
    url = f"https://api.telegram.org/bot{TOKEN}/sendMessage"
    payload = {"chat_id": CHAT_ID, "text": message, "parse_mode": "Markdown"}
    requests.post(url, json=payload)

def get_market_data(symbol):
    df = yf.download(symbol, period="5d", interval="1m", progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.droplevel(1)
    return df

def analyze_pro_market(df):
    close = df['Close']
    open_p = df['Open']

    # ১. RSI হিসাব (১৪ পিরিয়ড)
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -1 * delta.clip(upper=0)
    avg_gain = gain.rolling(window=14).mean()
    avg_loss = loss.rolling(window=14).mean()
    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    current_rsi = rsi.iloc[-1]

    # ২. বোলিংগার ব্যান্ড হিসাব (২০ পিরিয়ড)
    sma20 = close.rolling(window=20).mean()
    std20 = close.rolling(window=20).std()
    upper_band = sma20 + (std20 * 2)
    lower_band = sma20 - (std20 * 2)

    # ৩. ক্যান্ডেলস্টিক কনফার্মেশন (সবুজ বা লাল ক্যান্ডেল)
    current_close = close.iloc[-1]
    current_open = open_p.iloc[-1]
    is_bullish = current_close > current_open
    is_bearish = current_close < current_open

    signal = None
    if current_rsi < 30 and current_close <= lower_band.iloc[-1] and is_bullish:
        signal = "UP"
    elif current_rsi > 70 and current_close >= upper_band.iloc[-1] and is_bearish:
        signal = "DOWN"

    return signal, current_rsi

print("Multi-Market Pro Trader Bot is running...")
last_signals = {symbol: None for symbol in SYMBOLS}
last_signal_times = {symbol: 0 for symbol in SYMBOLS}
cooldown_period = 900  # একই পেয়ারে ১৫ মিনিট বিরতি

while True:
    for symbol in SYMBOLS:
        try:
            df = get_market_data(symbol)
            current_signal, current_rsi = analyze_pro_market(df)
            current_time = time.time()

            if current_signal:
                if current_signal != last_signals[symbol] or (current_time - last_signal_times[symbol] > cooldown_period):
                    if current_signal == "UP":
                        signal_msg = f"🟢 *PRO UP (CALL) Signal*\nMarket: {symbol}\n✅ RSI Oversold: {current_rsi:.2f}\n✅ Bollinger & Candlestick Confirmed!"
                    else:
                        signal_msg = f"🔴 *PRO DOWN (PUT) Signal*\nMarket: {symbol}\n✅ RSI Overbought: {current_rsi:.2f}\n✅ Bollinger & Candlestick Confirmed!"

                    send_telegram_signal(signal_msg)
                    print(f"Signal sent for {symbol}: {signal_msg}")

                    last_signals[symbol] = current_signal
                    last_signal_times[symbol] = current_time

            elif not current_signal:
                last_signals[symbol] = None

        except Exception as e:
            print(f"Error analyzing {symbol}:", e)

        time.sleep(10) # পেয়ারগুলোর মাঝে সামান্য বিরতি

    time.sleep(60) # পুরো সাইকেল শেষ হওয়ার পর ১ মিনিট অপেক্ষা


Multi-Market Pro Trader Bot is running...


/tmp/ipykernel_3548/373186336.py:21: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(symbol, period="5d", interval="1m", progress=False)
/tmp/ipykernel_3548/373186336.py:21: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(symbol, period="5d", interval="1m", progress=False)
/tmp/ipykernel_3548/373186336.py:21: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(symbol, period="5d", interval="1m", progress=False)
